In [1]:
from sentence_transformers import SentenceTransformer
import json
import os
import pandas as pd
import numpy as np
import faiss

/home/alex/Documents/Arxiv-RAG-Project/rag_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('dataset/arxiv_nlp.csv')
df.shape

(101840, 5)

In [3]:
EMBEDDING_MODEL = SentenceTransformer("all-miniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4222.22it/s]


In [4]:
def vectorize_text(text):
    embeddings = EMBEDDING_MODEL.encode(text, show_progress_bar=True)

    return np.array(embeddings).astype('float32')

In [9]:
user_input = input()
user_embeddings = vectorize_text([user_input])

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.02it/s]


In [6]:
INDEX_PATH = 'index/pdf_nlp_chunks.index'
index = faiss.read_index(INDEX_PATH)

In [ ]:
# D = similarity
# I = indexes
D, I = index.search(user_embeddings, 4)

In [17]:
print(I)

[[3787167 4056566 4081247 4338836 4165566 3511791 3593762 3593763 4050617
  3970247]]


In [ ]:
import sqlite3
def fetch_sqlite(db_path, target_index):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    placeholders = ','.join('?' * len(target_index))

    index = tuple(int(idx) for idx in target_index)
    query = f'SELECT text FROM chunks WHERE id IN ({placeholders})'
    cursor.execute(query, index)

    res = cursor.fetchall()
    conn.close()

    return [row[0] for row in res]

context_list = fetch_sqlite('dataset/db_chunks.sqlite', I[0])
    